# Metropolis algorithm

## What is autocorrelation time?

Autocorrelation time quantifies how long a system remembers its past — in other words, how many steps it takes for correlations between successive states to decay. In computational physics and statistical mechanics, this concept is essential for understanding how efficiently an algorithm samples the system’s equilibrium distribution.

There are two main ways to estimate the autocorrelation time:

1. Theoretically (heuristically) — by analyzing the physical processes or algorithmic mechanisms that govern relaxation (e.g., diffusion, collective motion, or energy barriers). This helps identify what slows down convergence but may miss hidden slow modes, leading to an underestimate.

2. Empirically — by computing the autocorrelation function of several physical observables (like energy, magnetization, or particle density). If none of these observables couple strongly to the slowest mode, the true autocorrelation time may again be underestimated. Therefore: there
is always the danger that our **chosen set of observables has failed to include one that has strong enough overlap with the slowest mode leading to an underestimate**.

The integrated autocorrelation time, often denoted $\tau_{\text{int}}$, determines the effective number of independent samples collected. Physically, it corresponds to how fast the system relaxes toward equilibrium — thus linking the algorithm’s sampling efficiency to actual dynamical or collective phenomena in the model. For example, near a critical point, long-range correlations in physical quantities manifest as very long autocorrelation times, a phenomenon known as **critical slowing down**.

The actual rate of convergence to equilibrium from a given initial distribution may be much faster than the worst-case estimate given by $\tau_{\text{exp}}$. Thus, it is usual to determine empirically when "equilibrium" has been achieved, by plotting selected observables as a function of time and noting when the initial transient appears to end.

Watch out for **metastability**!
For example, near a first-order phase transition, most Monte Carlo methods suffer from metastability associated with transitions between configurations typical of the distinct pure phases. We can try initial conditions typical of each of these phases (e.g., for many models, a "hot" start (random initial configuration) and a "cold" start (ordered initial start; all spins up or all spins down)). Consistency between these runs does not guarantee
that metastability is absent, but it does give increased confidence. Plots of observables as a function of time are also useful indicators of possible metastability.


**Once we estimate how long it takes to reach equilibrium**, we simply **discard (“burn in”) the initial portion of the data** — up to a cutoff time $n_{\text{disc}}$.

Although this is not strictly necessary in the asymptotic limit (since systematic errors from the transient decay faster than statistical ones), in practice, keeping non-equilibrium data can introduce significant bias if the starting configuration is far from equilibrium. Discarding the transient data therefore helps ensure that only equilibrium samples contribute to averages, improving the accuracy of estimated physical quantities.

Even in **finite-size lattices**, a cutoff is required. The only difference is that, since the system is finite, its equilibration and correlation times are in principle bounded. Still, one must determine an appropriate thermalization time for each lattice size (often a few times the autocorrelation time of the slowest mode) before starting data collection.

For the 2D Ising model, we discard approximately $20\tau$ of initial data to remove initialization bias and run for at least $1000\tau$ to ensure percent-level statistical accuracy. These ratios hold in general, though the autocorrelation time $\tau$ increases strongly near the critical temperature due to critical slowing down and depends on both lattice size and simulation algorithm.

## Metropolis Algorithm explained


1. Initialization:
- Create an $L \times L$ lattice of spins, either randomly (each spin up or down with equal probability) or uniformly aligned (all spins up).
- Compute the initial total energy and magnetization of the system.

2. Metropolis update loop:
Repeat for many steps (I did 600 000 × L² total spin flips):

- Select a random spin on the lattice.
- Compute the change in energy ($\Delta E$) that would occur if that spin were flipped, considering its four nearest neighbors and an external field $h$.

- Apply the Metropolis acceptance criterion:
    - If $\Delta E \le 0$, flip the spin (energy decreases).
    - If $\Delta E > 0$, flip the spin with probability $e^{-\beta \Delta E}$, where $\beta = 1 / (k_B T)$.

- Update the total energy, magnetization, and count accepted flips.

3. Data collection:
- Record the total energy and magnetization at each step.

- Compute and report the acceptance rate (fraction of accepted spin flips).

4. Post-processing and visualization:

- Save energies, magnetizations, and final spin configurations for each temperature (i.e., each value of $\beta$).

5. Temperature scan:
- The procedure repeats for multiple $\beta$ values, covering a range of temperatures to study how the system transitions (i.e. near the critical temperature around $\beta \approx 0.4407$ for the 2D Ising model).

## Red-Black Metropolis Single-Spin Update

The checkerboard (red–black) updating scheme used in the Ising Metropolis algorithm originates from mid‑20th‑century Gauss–Seidel numerical iteration methods and was adopted in Ising Monte Carlo simulations in the 1970s–1980s to enable parallel, non‑conflicting spin updates while preserving detailed balance.

In the Ising model, each spin interacts with its four neighbors (in 2D).

If you color the grid in a checkerboard pattern:


- Each “black” spin only interacts with “white” spins.

- Each “white” spin only interacts with “black” spins.

Therefore:


- You can update all black spins using the current white spins.

- Then update all white spins using the just‑updated black spins.

This preserves the detailed balance condition while allowing the possibility of partial parallelism, because updates within a parity set do not interfere.

Sources:

https://www.sciencedirect.com/science/article/pii/S0375960124004663

https://inspirehep.net/literature/883697

## Comparison of algorithms

The better algorithm is the one that has the smaller autocorrelation time, when time is measured in units of computer (CPU) time.